# Semana 5 · Cierre de fundamentos — consolidación de la escena y primer commit con Git

**Electiva Profesional I (El Metaverso) · 2026-II**

hasta ahora, tenemos una galería como datos (S3), esa misma galería como `Mesh` reales en `pythreejs` (S4) y una escena que corre en el navegador sin instalar nada (S4, WebXR). Ahora, procedemos a efectuar dos tareas adicionales:

- **Consolidar.** Las posiciones repartidas entre celdas de S3 y S4 se juntan en **una sola estructura de datos**: el *estado del mundo* — el lenguaje de entidad/atributo/estado que la Unidad 2 va a usar para modelos computacionales. Se verifica que ese estado reproduce, número por número, lo que ya se había construido — no una escena nueva.
- **Miércoles — versionar: **Git**. Usted ya tiene tiene cuatro semanas de cuadernos que puede perder fácilmente y ese riesgo ya es real.

Ejecute de arriba abajo. En **`[Trabajo del estudiante]`** complete antes de seguir; las demás celdas ya están resueltas y explicadas.


## 0. Entorno

Mismo entorno `mv2026` de las semanas 1 a 4. Hoy se suma una verificación: ¿Está `git` disponible?

In [1]:
import sys
import subprocess
import numpy as np

print('Ejecutable (debe mencionar mv2026):', sys.executable)
print('numpy:', np.__version__)

try:
    import pythreejs as p3
    TENGO_3D = True
    print('pythreejs: OK')
except Exception as e:
    TENGO_3D = False
    print('pythreejs: no disponible ->', e)

try:
    salida = subprocess.run(['git', '--version'], capture_output=True, text=True, check=True)
    print(salida.stdout.strip())
except Exception as e:
    print('git: no disponible ->', e)
    print('Instálelo dentro del entorno con: conda install -c conda-forge git')


Ejecutable (debe mencionar mv2026): C:\Users\loghan\miniconda3\envs\mv2026\python.exe
numpy: 2.5.1
pythreejs: OK
git version 2.55.0.windows.1


---
## 1. La escena como estado del mundo

Hasta hoy, la posición de cada obra se encuentra repartida: como tupla en `galeria`, como argumento de `p3.Mesh(...)` y otra vez como atributo HTML en `escena_webxr.html`. Tres lugares para el mismo número es exactamente lo que se denomina como **falta de una sola fuente de verdad**.

Hoy esas posiciones se consolidan en **una sola estructura**: un diccionario `escena_base` con `sala`, `pared`, la lista de `obras` (cada una con su posición original, la posición reubicada y su color) y la `escultura`. Es exactamente el vocabulario que la Unidad 2 usa para un modelo: **entidades** (obras, escultura), **atributos** (posición, color) y el **estado** del mundo como un solo objeto que se puede imprimir, guardar o simular — sin abrir `pythreejs` para saber dónde está algo.


In [2]:
escena_base = {
    'sala': {
        'tipo': 'piso',
        'posicion': [5.0, 0.0, 0.0],
        'dimensiones': {'width': 40, 'height': 20},
    },
    'pared': {
        'posicion': [0.0, 2.0, -4.0],
        'dimensiones': {'width': 14, 'height': 4, 'depth': 0.2},
    },
    'obras': [
        {'nombre': 'El Reflejo Digital', 'posicion_original': [2.0, 0.0, 5.0],
         'posicion_actual': [13.0, 0.0, 2.5], 'color': '#00ff00'},
        {'nombre': 'Fragmentos', 'posicion_original': [-3.0, 1.5, 2.0],
         'posicion_actual': [10.5, 0.75, 1.0], 'color': '#ccff00'},
        {'nombre': 'Eco', 'posicion_original': [5.0, 0.0, 1.0],
         'posicion_actual': [14.5, 0.0, 0.5], 'color': '#008080'},
    ],
    'escultura': {'nombre': 'Punto de Fuga', 'posicion': [2.12, 0.0, 2.12]},
    'camara': {'posicion': [10, 8, 14], 'fov': 50},
    'luces': {
        'direccional': {'posicion': [5, 8, 5], 'intensidad': 1.0},
        'ambiente': {'intensidad': 0.4},
    },
}

for obra in escena_base['obras']:
    print(f"{obra['nombre']:<20} original {obra['posicion_original']} -> actual {obra['posicion_actual']}")


El Reflejo Digital   original [2.0, 0.0, 5.0] -> actual [13.0, 0.0, 2.5]
Fragmentos           original [-3.0, 1.5, 2.0] -> actual [10.5, 0.75, 1.0]
Eco                  original [5.0, 0.0, 1.0] -> actual [14.5, 0.0, 0.5]


### 🅢 Punto de encuentro — entidad, atributo, estado (adelanto de la Unidad 2)

- **Entidad** = algo que existe en el mundo y que se puede nombrar (`"El Reflejo Digital"`, la   `escultura`).
- **Atributo** = un dato que describe a esa entidad (`posicion_actual`, `color`).
- **Estado del mundo** = el conjunto de todas las entidades con sus atributos, **en un instante dado** — hoy ese instante es "quieto"; posteriormente, vamos a hacer que cambie con el tiempo. `escena_base` de hoy es la versión más simple posible de un estado: no se mueve todavía.


---
## 2. Reproducibilidad: ¿La consolidación da los mismos números que la semana 4?

Consolidar no es "escribir de nuevo lo que ya existía" — es demostrar que la nueva estructura **reproduce exactamente** los números que ya estaban verificados. Si `escena_base` tuviera un error de transcripción, ningún `pythreejs` se quejaría (los números son solo datos): el error quedaría invisible hasta que alguien comparara contra la fuente. Por eso se verifica con `assert`, no a simple vista.

**Demostrado — "El Reflejo Digital":**


In [3]:
factor_escala = 0.5
destino_sala = np.array([12.0, 0.0, 0.0])

def mover_correcto(posicion):
    return (np.array(posicion) * factor_escala) + destino_sala

reflejo = next(o for o in escena_base['obras'] if o['nombre'] == 'El Reflejo Digital')
recalculado = mover_correcto(reflejo['posicion_original'])
assert np.allclose(recalculado, reflejo['posicion_actual']), 'La posición consolidada no coincide con la S4'
print('El Reflejo Digital: OK ->', recalculado.round(2))


El Reflejo Digital: OK -> [13.   0.   2.5]


**`[Trabajo del estudiante]`** Repita la misma verificación para `"Fragmentos"` y `"Eco"`: recalcule
con `mover_correcto` a partir de `posicion_original` y compruebe con `assert` que coincide con
`posicion_actual`. Si algún `assert` falla, **no lo borre ni lo comente** — es la señal de que
`escena_base` tiene un error de transcripción; corríjalo ahí, no en la verificación.

In [4]:
# [Trabajo del estudiante] verifique 'Fragmentos' y 'Eco' con el mismo patrón que 'El Reflejo Digital'.
fragmentos = next(o for o in escena_base['obras'] if o['nombre'] == 'Fragmentos')
recalculado_frag = mover_correcto(fragmentos['posicion_original'])
assert np.allclose(recalculado_frag, fragmentos['posicion_actual']), 'La posición consolidada no coincide con la S4'
print('Fragmento: OK ->', recalculado.round(2))

eco = next(o for o in escena_base['obras'] if o['nombre'] == 'Eco')
recalculado_eco = mover_correcto(eco['posicion_original'])
assert np.allclose(recalculado_eco, eco['posicion_actual']), 'La posición consolidada no coincide con la S4'
print('Eco: OK ->', recalculado.round(2))

Fragmento: OK -> [13.   0.   2.5]
Eco: OK -> [13.   0.   2.5]


---
## 3. Verlo de nuevo: `pythreejs` construido a partir de `escena_base`

La escena de la sesión pasada se armó a mano, pieza por pieza, en el cuaderno. Esta se arma **recorriendo `escena_base`** — si mañana cambia una posición en el diccionario, la escena cambia con ella, sin tocar el código que la dibuja. Esa separación (datos por un lado, dibujo por otro) es la que hace posible, más adelante, simular un mundo sin reescribir cómo se ve.


In [5]:
if TENGO_3D:
    geometria_piso = p3.PlaneGeometry(width=escena_base['sala']['dimensiones']['width'],
                                       height=escena_base['sala']['dimensiones']['height'])
    geometria_piso.exec_three_obj_method('rotateX', -np.pi / 2)
    piso = p3.Mesh(geometry=geometria_piso,
                    material=p3.MeshStandardMaterial(color='#b9b0a0'),
                    position=escena_base['sala']['posicion'])

    d = escena_base['pared']['dimensiones']
    pared = p3.Mesh(geometry=p3.BoxGeometry(width=d['width'], height=d['height'], depth=d['depth']),
                     material=p3.MeshStandardMaterial(color='#e4ded0'),
                     position=escena_base['pared']['posicion'])

    obras_mesh = [
        p3.Mesh(geometry=p3.BoxGeometry(width=1.0, height=1.0, depth=0.1),
                material=p3.MeshStandardMaterial(color=o['color']),
                position=o['posicion_actual'])
        for o in escena_base['obras']
    ]

    escultura = p3.Mesh(geometry=p3.SphereGeometry(radius=0.5, widthSegments=20, heightSegments=20),
                         material=p3.MeshStandardMaterial(color='#9b59b6'),
                         position=escena_base['escultura']['posicion'])

    camara = p3.PerspectiveCamera(position=escena_base['camara']['posicion'], fov=escena_base['camara']['fov'])
    luz = p3.DirectionalLight(color='white',
                               intensity=escena_base['luces']['direccional']['intensidad'],
                               position=escena_base['luces']['direccional']['posicion'])
    ambiente = p3.AmbientLight(intensity=escena_base['luces']['ambiente']['intensidad'])

    escena = p3.Scene(children=[piso, pared, *obras_mesh, escultura, camara, luz, ambiente])
    render = p3.Renderer(scene=escena, camera=camara,
                          controls=[p3.OrbitControls(controlling=camara)],
                          width=700, height=450)
    display(render)
    print('Escena reconstruida desde escena_base: piso, pared, 3 obras, escultura, cámara, 2 luces.')
else:
    print('pythreejs no disponible en este equipo. Reconstrucción confirmada solo con los datos:')
    for o in escena_base['obras']:
        print(' ', o['nombre'], '->', o['posicion_actual'])


Renderer(camera=PerspectiveCamera(position=(10.0, 8.0, 14.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, …

Escena reconstruida desde escena_base: piso, pared, 3 obras, escultura, cámara, 2 luces.


**Autoevaluación.** ¿Podría otra persona reconstruir esta escena solo con `escena_base` y esta celda, sin haber visto la sesión previa? Si su respuesta es "no, porque además necesitaría...", anote qué falta — eso es exactamente lo que un cuaderno reproducible no puede dejar implícito.

Si, otra persona podría reconstruir y ejecutar la escena utilizando únicamente escena_base y esa celda.

Para que el cuaderno sea verdaderamente reproducible y autónomo, falta hacer explícito lo siguiente:

La importación de librerías, lo que pasa es que no entenderia que teine el codigo pero si lo podria ejecutar normalmente


---
## 4. Por qué versionar: el problema que Git resuelve

Ya tiene cuatro cuadernos y el de hoy. Ninguno está protegido: un `Kernel → Restart`, un archivo sobrescrito sin querer o simplemente "ayer funcionaba y hoy no sé qué cambié" — y no hay forma de volver atrás ni de saber qué era distinto. Eso es exactamente el problema que un sistema de control de versiones resuelve: **guardar fotografías con fecha de su carpeta de trabajo**, para poder compararlas o volver a cualquiera de ellas.

**Git no se usa desde el cuaderno.** A partir de aquí, abra **PowerShell** en la carpeta donde tiene los cuadernos del curso y siga el guion de la sesión de hoy. Vuelva a este cuaderno solo para las dos últimas celdas (la evidencia).


## 5. `git init`, `.gitignore` y el primer commit (en PowerShell, no aquí)

Resumen de lo que hace en la sesión:

1. `git init` en la carpeta raíz de sus materiales del curso.
2. Crear un archivo `.gitignore` con estas dos líneas, **antes** del primer `git add`:
   ```
   .ipynb_checkpoints/
   __pycache__/
   ```
3. `git add .` y `git commit -m "primer commit: cuadernos S2-S5"`.
4. Un segundo commit más pequeño (por ejemplo, después de completar el `[Trabajo del estudiante]` de la sección 2 de este mismo cuaderno): `git add cierre_fundamentos.ipynb` y `git commit -m "consolidación: verificación de Fragmentos y Eco"`.
5. `git log --oneline` para ver el historial.


### El momento "no da error pero está mal" de esta semana

Alguien hace `git add .` **antes** de escribir el `.gitignore`. Git no se queja: agrega con la misma tranquilidad `cierre_fundamentos.ipynb` y también
`.ipynb_checkpoints/cierre_fundamentos-checkpoint.ipynb`, `__pycache__/galeria_utils.cpython-312.pyc` y cualquier copia de respaldo que Jupyter haya dejado tirada. El commit **funciona**, `git log` se ve perfectamente normal y nada indica que algo salió mal — hasta que el repositorio pesa el doble de lo que debería, o dos "archivos" que en realidad son el mismo cuaderno aparecen como si fueran contenido distinto para revisar. Ningún error de Python ni de Git avisa: el `.gitignore`
tardío no borra lo ya versionado, solo evita que seguir empeorando.

**`[Trabajo del estudiante]`** Si ya hizo `git add .` antes de crear el `.gitignore`, corríjalo ahora: `git rm -r --cached .ipynb_checkpoints __pycache__`, luego `git add .gitignore` y un commit que documente la limpieza (`git commit -m "gitignore: dejar de versionar checkpoints y cache"`). Si ya lo hizo bien desde el principio, dígalo en el informe de cierre.


---
## 6. Evidencia — pegue aquí el resultado real de su repositorio

Ejecute estas dos celdas **desde este mismo cuaderno**, ya con sus commits hechos en PowerShell
(Jupyter puede correr comandos de consola anteponiendo `!`). No copie un resultado de otra
persona: si el historial no es el suyo, no cuenta como evidencia.

In [6]:
# [Trabajo del estudiante] ejecute esta celda tal cual, después de sus commits.
!git log --oneline


c768c4f primer commit: cuadernos S2-S5


In [7]:
# [Trabajo del estudiante] ejecute esta celda tal cual. Confirme que NO aparecen
# archivos dentro de .ipynb_checkpoints/ ni __pycache__/ como "untracked" sin más,
# ni (peor) como parte de lo ya confirmado.
!git status


On branch master
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   cierre_fundamentos.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


---
## 7. Informe de cierre

En cuatro a seis frases: ¿Qué fue lo más confuso de consolidar `escena_base` frente a tenerlo repartido en varias celdas de semanas distintas? ¿Le pasó el error del `.gitignore` tardío, o lo evitó? Si tuviera que explicarle a alguien que nunca ha usado Git **por qué** un commit no es lo mismo que "guardar el archivo" ¿Qué le diría?

Lo más confuso de consolidar escena_base fue rastrear las dependencias implícitas y las mutaciones de estado que quedaron dispersas en celdas ejecutadas en distinto orden durante semanas previas. Respecto al control de versiones, logré evitar el error del .gitignore tardío porque creé el archivo con las reglas de exclusión antes de ejecutar el primer git add .. Para explicar la diferencia con Git, diría que "guardar un archivo" solo sobreescribe su texto actual en el disco duro para no perder el trabajo inmediato. En cambio, un commit es como tomar una fotografía con fecha y descripción de todo el proyecto en un punto clave del tiempo. Mientras que guardar simplemente reemplaza el presente, hacer un commit construye un historial de restauración exacto al que puedes regresar cuando lo necesites.

> **Documentar un intento que no cerró puntúa.** Si `git` no quedó instalado a tiempo, si un
> comando dio un error que no logró resolver, o si el `.gitignore` no alcanzó a limpiar el
> historial completo, escriba exactamente qué intentó y qué mensaje le dio la terminal — eso vale
> más que una entrega en blanco o que afirmar "ya quedó" sin la evidencia de la sección 6.
